In [12]:
import numpy as np
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt

# --- PARAMETERS ---
GRID_SIZE = 4
HIDDEN_DIM = 256     # Standard width
NUM_LAYERS = 3       # Standard depth (Input -> Hidden -> Hidden -> Output)
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
NUM_ITERATIONS = 10000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DATA (Keep the Augmentation - This is the secret sauce) ---
def get_symmetries(image_stack):
    syms = []
    curr = image_stack
    for _ in range(4):
        syms.append(curr)
        syms.append(np.flip(curr, axis=2))
        curr = np.rot90(curr, axes=(1, 2))
    return syms

def generate_robust_data(grid_size):
    full_pairs = []
    coords = [(r, c) for r in range(grid_size) for c in range(grid_size)]
    
    # Generate Base Data
    for p1 in coords:
        for p2 in coords:
            state = np.zeros((2, grid_size, grid_size), dtype=np.float32)
            state[0, p1[0], p1[1]] = 1.0
            state[1, p2[0], p2[1]] = 1.0
            dist = np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
            full_pairs.append((state, dist))

    # Split indices strictly
    random.shuffle(full_pairs)
    split = int(len(full_pairs) * 0.8)
    raw_train = full_pairs[:split]
    raw_test = full_pairs[split:]

    # Augment Training Data
    aug_train = []
    for state, dist in raw_train:
        symmetries = get_symmetries(state)
        for s in symmetries:
            aug_train.append((s.copy(), dist))
            
    return aug_train, raw_test

train_data, test_data = generate_robust_data(GRID_SIZE)

# --- 2. THE PURE MLP (No Residuals, No LayerNorm) ---
class PureMLP(nn.Module):
    def __init__(self, grid_size, hidden_dim, num_layers):
        super().__init__()
        input_dim = 2 * grid_size * grid_size
        
        layers = []
        # Input Layer
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        
        # Hidden Layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            
        # Output Layer
        layers.append(nn.Linear(hidden_dim, 1))
        
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

model = PureMLP(GRID_SIZE, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE) # Standard Adam
criterion = nn.MSELoss()

# --- 3. TRAIN ---
model.train()
losses = []

print(f"Training PURE MLP on {DEVICE}...")
for i in range(NUM_ITERATIONS):
    batch = random.sample(train_data, BATCH_SIZE)
    
    states = torch.tensor(np.stack([x[0] for x in batch])).to(DEVICE)
    targets = torch.tensor(np.array([x[1] for x in batch], dtype=np.float32)).unsqueeze(1).to(DEVICE)
    
    preds = model(states)
    loss = criterion(preds, targets)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if i % 2000 == 0:
        print(f"Iter {i}: Loss {loss.item():.6f}")

# --- 4. EVALUATE ---
model.eval()
test_states = torch.tensor(np.stack([x[0] for x in test_data])).to(DEVICE)
test_targets = np.array([x[1] for x in test_data])

with torch.no_grad():
    test_preds = model(test_states).cpu().numpy().flatten()

mae = np.mean(np.abs(test_targets - test_preds))
mask_pos = test_targets > 0.01
mape = np.mean(np.abs(test_targets[mask_pos] - test_preds[mask_pos]) / test_targets[mask_pos]) * 100

print(f"\n=== PURE MLP RESULTS ===")
print(f"MAE:  {mae:.5f}")
print(f"MAPE: {mape:.4f}%")

if mape < 1.0:
    print("\nCONCLUSION: Pure MLP works perfectly given Data Augmentation.")
else:
    print("\nCONCLUSION: Pure MLP struggled compared to ResMLP.")

Training PURE MLP on cuda...
Iter 0: Loss 4.233313
Iter 2000: Loss 0.000000
Iter 4000: Loss 0.000048
Iter 6000: Loss 0.000004
Iter 8000: Loss 0.000736

=== PURE MLP RESULTS ===
MAE:  0.02088
MAPE: 1.0079%

CONCLUSION: Pure MLP struggled compared to ResMLP.
